# Native three-dimensional Stokes, Brinkman and Oseen

Tetrahedral MHM uses continuous local velocity/pressure fields and independent vector traces on macrofaces. This notebook checks an affine patch and replays the four manufactured convergence studies; these are original 3D validation cases, not historical article meshes.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.flow3d import solve_flow_3d
from pymhm.tetrahedral import TetraMesh

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


In [ ]:
def velocity(x):
    """Divergence-free affine field with inhomogeneous boundary data."""
    return np.column_stack((x[:, 0], -x[:, 1], np.ones(len(x))))

def pressure(x):
    """Zero-mean affine pressure on the unit cube."""
    return x[:, 0] + x[:, 1] - 1

with threadpool_limits(1):
    patch = solve_flow_3d(TetraMesh.unit_cube(), dirichlet=velocity,
                          source=(1., 1., 0.), formulation="taylor-hood")
    errors = {"velocity": patch.l2_error(velocity),
              "pressure": patch.pressure_l2_error(pressure),
              "divergence": patch.divergence_l2()}
assert max(errors.values()) < 1e-10
errors


The pressure mean is a gauge because every boundary component prescribes velocity. Pseudotraction uses the grad-grad convention; the raw stress is not an H(div) reconstruction.

In [ ]:
folder = ROOT / "examples/results/flow3d"
campaign = json.loads((folder / "campaign.json").read_text())
rows = campaign["rows"]
assert len(rows) == 20
for row in rows:
    assert hashlib.sha256((folder / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
    assert row["max_error_quadrature_relative_difference"] < 1e-8
    assert row["backward_residual"] < 1e-8
for case in sorted({row["case"] for row in rows}):
    selected = sorted((row for row in rows if row["case"] == case),
                      key=lambda row: row["macro_subdivisions"])
    assert [row["macro_subdivisions"] for row in selected] == [1, 2, 3, 4, 5]
    assert all(selected[-1][key] < selected[0][key]
               for key in ("velocity_l2", "pressure_l2", "velocity_h1_seminorm"))
[(r["case"], r["macro_subdivisions"], r["velocity_l2"], r["pressure_l2"])
 for r in rows]


In [ ]:
figures = ROOT / "docs/figures/flow3d"
for name in ["convergence", "stokes-th2-fields", "brinkman-usfem1-fields",
             "brinkman-usfem2-fields", "oseen-p2-fields", "oseen-p2-components"]:
    display(Image(filename=str(figures / f"{name}.png")))


The slice panels retain one-sided fine-element values and the true macro boundaries. Error quadrature orders seven and eight are checked independently. The reported time is elapsed campaign time under concurrent activity, not a speedup measurement. This evidence covers the stated finite spaces and coefficients; it is not a uniform inf-sup or coefficient-robustness certificate.